# Model Training

## Imports

In [1]:
import torch
import numpy as np

from reionemu import (
    load_training_arrays,
    DataLoaderConfig,
    make_dataloaders,
    MCDropoutEmulator,
    FitConfig,
    fit,
    mse,
    rmse,
    physical_mean_relative_error,
)

## Paths & Constants

In [2]:
h5_path = "../data/condensed_v6.h5"
# Base Model
base_ckpt_path = "../checkpoints/base_model/checkpoint.pt"
base_norm_path = "../checkpoints/base_model/norm/"
base_split_idx_path = "../checkpoints/base_model/split_idx/"
# Larger Model
larger_ckpt_path = "../checkpoints/larger_model/checkpoint.pt"
larger_norm_path = "../checkpoints/larger_model/norm/"
larger_split_idx_path = "../checkpoints/larger_model/split_idx/"

In [3]:
SEED = 42

dlcfg = DataLoaderConfig(
    batch_size=32,
    seed=SEED,
    shuffle_train=True,
    normalize_X=True,
    normalize_Y=False,
)

fitcfg = FitConfig(
    epochs=1000,
    device="mps",
    early_stopping_patience=150,
    gradient_clipping=None,
    seed=SEED,
)

## Build Dataloaders

In [4]:
np.random.seed(SEED)
torch.manual_seed(SEED)

loaders, norms, ell = make_dataloaders(h5_path, split={"train": 0.70, "val": 0.10, "test": 0.20}, config=dlcfg)

In [5]:
train_idx = loaders["train"].dataset.indices
val_idx = loaders["val"].dataset.indices
test_idx = loaders["test"].dataset.indices

np.savez(base_split_idx_path + "split_idx", train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)
np.savez(larger_split_idx_path + "split_idx", train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)

# Base Model
---

In [6]:
np.random.seed(SEED)
torch.manual_seed(SEED)

## Build Model

In [7]:
modelcfg = {
    "input_dim": 4,
    "output_dim": 5,
    "hidden_dim": 20,
    "num_hidden_layers": 2,
    "activation": "relu",
    "dropout_rate": 0.1,
}
learning_rate = 1e-3
weight_decay = 1e-5

model = MCDropoutEmulator(**modelcfg)
lossfn = torch.nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

## Train Model

In [8]:
model = MCDropoutEmulator(**modelcfg)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
lossfn = torch.nn.MSELoss()

train_results = fit(
    model,
    loaders["train"],
    loaders["val"],
    optimizer,
    lossfn,
    fitcfg,
    metrics={"Physical Mean Relative Error": physical_mean_relative_error, "RMSE": rmse, "MSE": mse},
    evaluation="evaluate_mc_metrics",
    n_mc_samples=201,
)

Epoch 001: train=0.331176, val=0.327115, Physical Mean Relative Error=0.640767, RMSE=0.568223, MSE=0.327115, predictive_std=0.050195
Epoch 002: train=0.276448, val=0.283668, Physical Mean Relative Error=0.574435, RMSE=0.528351, MSE=0.283668, predictive_std=0.057999
Epoch 003: train=0.238637, val=0.245549, Physical Mean Relative Error=0.510569, RMSE=0.490913, MSE=0.245549, predictive_std=0.067792
Epoch 004: train=0.199248, val=0.197244, Physical Mean Relative Error=0.429010, RMSE=0.438958, MSE=0.197244, predictive_std=0.080755
Epoch 005: train=0.146726, val=0.132602, Physical Mean Relative Error=0.315805, RMSE=0.358344, MSE=0.132602, predictive_std=0.101638
Epoch 006: train=0.094648, val=0.072394, Physical Mean Relative Error=0.210343, RMSE=0.263204, MSE=0.072394, predictive_std=0.126285
Epoch 007: train=0.061091, val=0.042694, Physical Mean Relative Error=0.154863, RMSE=0.201633, MSE=0.042694, predictive_std=0.139165
Epoch 008: train=0.050082, val=0.030838, Physical Mean Relative Error

## Save Checkpoint

In [9]:
torch.save(model.state_dict(), base_ckpt_path)
np.save(base_norm_path + "X_mean.npy", norms["X"].mean)
np.save(base_norm_path + "X_std.npy", norms["X"].std)
np.save(base_norm_path + "ell.npy", np.asarray(ell))

# Larger Model
---

In [10]:
np.random.seed(SEED)
torch.manual_seed(SEED)

## Build Model

In [11]:
modelcfg = {
    "input_dim": 4,
    "output_dim": 5,
    "hidden_dim": 32,
    "num_hidden_layers": 3,
    "activation": "relu",
    "dropout_rate": 0.1,
}
learning_rate = 1e-3
weight_decay = 1e-5

model = MCDropoutEmulator(**modelcfg)
lossfn = torch.nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

## Train Model

In [12]:
model = MCDropoutEmulator(**modelcfg)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
lossfn = torch.nn.MSELoss()

train_results = fit(
    model,
    loaders["train"],
    loaders["val"],
    optimizer,
    lossfn,
    fitcfg,
    metrics={"Physical Mean Relative Error": physical_mean_relative_error, "RMSE": rmse, "MSE": mse},
    evaluation="evaluate_mc_metrics",
    n_mc_samples=201,
)

Epoch 001: train=0.323709, val=0.312864, Physical Mean Relative Error=0.624932, RMSE=0.555275, MSE=0.312864, predictive_std=0.034630
Epoch 002: train=0.238310, val=0.193586, Physical Mean Relative Error=0.421577, RMSE=0.437534, MSE=0.193586, predictive_std=0.066232
Epoch 003: train=0.131015, val=0.062583, Physical Mean Relative Error=0.203068, RMSE=0.249179, MSE=0.062583, predictive_std=0.109390
Epoch 004: train=0.059497, val=0.025577, Physical Mean Relative Error=0.120281, RMSE=0.158130, MSE=0.025577, predictive_std=0.134070
Epoch 005: train=0.045075, val=0.019438, Physical Mean Relative Error=0.104829, RMSE=0.137657, MSE=0.019438, predictive_std=0.133631
Epoch 006: train=0.039583, val=0.020023, Physical Mean Relative Error=0.106286, RMSE=0.139402, MSE=0.020023, predictive_std=0.125386
Epoch 007: train=0.037100, val=0.018164, Physical Mean Relative Error=0.100733, RMSE=0.132932, MSE=0.018164, predictive_std=0.127614
Epoch 008: train=0.033328, val=0.015968, Physical Mean Relative Error

## Save Checkpoint

In [13]:
torch.save(model.state_dict(), larger_ckpt_path)
np.save(larger_norm_path + "X_mean.npy", norms["X"].mean)
np.save(larger_norm_path + "X_std.npy", norms["X"].std)
np.save(larger_norm_path + "ell.npy", np.asarray(ell))